In [1]:
import sqlite3
import pandas as pd

def luu_du_lieu_thuc_nghiem(db_name, X_test_raw, y_test, y_pred_knn, y_pred_dt, models_summary):
    """
    Hàm độc lập thực hiện kết nối SQLite, khởi tạo cấu trúc bảng 
    và lưu trữ toàn bộ kết quả dự đoán cùng hiệu năng mô hình.
    """
    # 1. Kết nối cơ sở dữ liệu
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    # 2. Làm sạch và khởi tạo các bảng
    cursor.execute("DROP TABLE IF EXISTS du_doan")
    cursor.execute("DROP TABLE IF EXISTS xac_suat")

    cursor.execute("""
    CREATE TABLE du_doan (
        ID                  INTEGER PRIMARY KEY AUTOINCREMENT,
        ApplicantIncome     REAL,
        CoapplicantIncome   REAL,
        LoanAmount          REAL,
        Loan_Amount_Term    REAL,
        Credit_History      REAL,
        Property_Area       INTEGER,
        KetQuaThucTe        INTEGER,
        DuDoan_KNN          INTEGER,
        DuDoan_DT           INTEGER,
        KNN_Dung            INTEGER,
        DT_Dung             INTEGER
    )
    """)

    cursor.execute("""
    CREATE TABLE xac_suat (
        ID          INTEGER PRIMARY KEY AUTOINCREMENT,
        TenMoHinh   TEXT,
        ThamSo      TEXT,
        Loai        TEXT,
        Accuracy    REAL,
        F1_Score    REAL
    )
    """)
    conn.commit()
    print('✅ [SQL] Đã khởi tạo cấu trúc 2 bảng: du_doan + xac_suat')

    # 3. Xử lý dữ liệu hồ sơ dự đoán bằng Pandas
    df_result = X_test_raw.copy().reset_index(drop=True)
    df_result['KetQuaThucTe'] = y_test.values
    df_result['DuDoan_KNN']   = y_pred_knn
    df_result['DuDoan_DT']    = y_pred_dt
    df_result['KNN_Dung'] = (y_pred_knn == y_test.values).astype(int)
    df_result['DT_Dung']  = (y_pred_dt  == y_test.values).astype(int)

    # 4. Đẩy dữ liệu chi tiết hồ sơ vào bảng 'du_doan'
    for _, row in df_result.iterrows():
        cursor.execute("""
            INSERT INTO du_doan VALUES (NULL,?,?,?,?,?,?,?,?,?,?,?)
        """, (
            float(row['ApplicantIncome']),
            float(row['CoapplicantIncome']),
            float(row['LoanAmount']),
            float(row['Loan_Amount_Term']),
            float(row['Credit_History']),
            int(row['Property_Area']),
            int(row['KetQuaThucTe']),
            int(row['DuDoan_KNN']),
            int(row['DuDoan_DT']),
            int(row['KNN_Dung']),
            int(row['DT_Dung'])
        ))

    # 5. Đẩy dữ liệu tổng hợp hiệu năng vào bảng 'xac_suat'
    for row in models_summary:
        cursor.execute("INSERT INTO xac_suat VALUES (NULL,?,?,?,?,?)", row)

    conn.commit()
    print(f'✅ [SQL] Đã lưu thành công {len(df_result)} hồ sơ vào bảng du_doan')
    print(f'✅ [SQL] Đã lưu thành công {len(models_summary)} lịch sử mô hình vào bảng xac_suat')

    # 6. Thực hiện các truy vấn báo cáo phân tích
    print('\n================ BÁO CÁO TRUY VẤN SQL ================')
    
    print('\n--- 1. Accuracy & F1-Score từng mô hình ---')
    print(pd.read_sql("""
        SELECT TenMoHinh, ThamSo, Loai,
               ROUND(Accuracy*100,2) as Accuracy,
               ROUND(F1_Score*100,2) as F1_Score
        FROM xac_suat
    """, conn).to_string(index=False))

    print('\n--- 2. Phân tích các hồ sơ nhiễu (Cả 2 mô hình đều đoán sai) ---')
    df_ca2_sai = pd.read_sql("SELECT * FROM du_doan WHERE KNN_Dung=0 AND DT_Dung=0", conn)
    print(f'Số lượng hồ sơ lỗi lọt lưới: {len(df_ca2_sai)} hồ sơ.')

    print('\n--- 3. Đánh giá Accuracy theo lát cắt thuộc tính Credit_History ---')
    print(pd.read_sql("""
        SELECT Credit_History, COUNT(*) as TongHoSo,
               ROUND(AVG(CAST(DT_Dung  AS FLOAT))*100, 1) as DT_Accuracy,
               ROUND(AVG(CAST(KNN_Dung AS FLOAT))*100, 1) as KNN_Accuracy
        FROM du_doan
        GROUP BY Credit_History
    """, conn).to_string(index=False))
    print('======================================================\n')

    # 7. Đóng kết nối
    conn.close()
    print(f'✅ Đã khóa dữ liệu và lưu file {db_name} hoàn tất!\n')

In [2]:
# 1. Tổng hợp kết quả tổng quan của tất cả các mô hình đã chạy
danh_sach_mo_hinh = [
    ('KNN Tự Code',   f'K={k_p3}',           'Tự cài đặt', acc_p3,  f1_p3),
    ('KNN Sklearn',   f'K={k_p3}',           'Thư viện',   acc_k3,  f1_k3),
    ('KNN Sklearn',   f'K={best_k}',         'Thư viện',   acc_knn, f1_knn),
    ('Decision Tree', f'depth={best_depth}', 'Thư viện',   acc_dt,  f1_dt),
]

# 2. Gọi hàm thực thi độc lập
luu_du_lieu_thuc_nghiem(
    db_name='QuanLyTinDung.db',
    X_test_raw=X_test_raw,
    y_test=y_test,
    y_pred_knn=y_pred_knn,
    y_pred_dt=y_pred_dt,
    models_summary=danh_sach_mo_hinh
)

NameError: name 'k_p3' is not defined